# End-to-End Text Classification: From TF-IDF to Deep Learning

---

## Learning Objectives

By the end of this project, you will:

- Build a complete text classification pipeline from raw text to predictions
- Implement text preprocessing using a reusable utility module
- Train traditional ML baselines (TF-IDF + Logistic Regression, TF-IDF + SVM)
- Build and train a deep learning model (Embedding + LSTM) in PyTorch
- Evaluate and compare all approaches with accuracy, F1, and confusion matrices
- Perform error analysis to understand model failure modes
- Decide when traditional ML suffices vs. when deep learning adds value

## Prerequisites

- **DL100**: Neural network fundamentals
- **DL150**: PyTorch foundations (tensors, `nn.Module`, optimizers, DataLoaders)
- **DL200**: MLP practical (training loops, evaluation)
- **DL400**: RNN/LSTM fundamentals
- **DL500**: NLP text preprocessing basics
- Basic familiarity with scikit-learn

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Dataset: 20 Newsgroups (4-Category Subset)](#2.-Dataset)
3. [Text Preprocessing Pipeline](#3.-Text-Preprocessing-Pipeline)
4. [Baseline 1: TF-IDF + Logistic Regression](#4.-Baseline-1:-TF-IDF-+-Logistic-Regression)
5. [Baseline 2: TF-IDF + SVM](#5.-Baseline-2:-TF-IDF-+-SVM)
6. [Deep Model: Embedding + LSTM Classifier](#6.-Deep-Model:-Embedding-+-LSTM)
7. [Evaluation: All Models](#7.-Evaluation)
8. [Comparison Table](#8.-Comparison-Table)
9. [Error Analysis](#9.-Error-Analysis)
10. [Conclusions: Traditional vs. Deep Learning](#10.-Conclusions)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")
from src.utils.seed import set_global_seed
set_global_seed(42)

from src.utils.device import get_device
from src.utils.training import EarlyStopping
from src.utils.metrics import compute_accuracy, compute_classification_report
from src.utils.plotting import plot_confusion_matrix
from src.utils.text_preprocessing import (
    normalize_text, basic_tokenize, build_vocab, texts_to_sequences
)

import numpy as np
import matplotlib.pyplot as plt
import time
import os
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split

device = get_device()
print("Setup complete.")

---

## 2. Dataset

We use the **20 Newsgroups** dataset, restricted to 4 categories for clarity:

| Category | Description |
|----------|-------------|
| `sci.space` | Space and astronomy discussions |
| `rec.sport.hockey` | Hockey discussions |
| `comp.graphics` | Computer graphics topics |
| `talk.politics.mideast` | Middle East politics |

These categories are thematically distinct, which makes initial classification feasible, yet some overlap exists in vocabulary.

In [ ]:
CATEGORIES = [
    "sci.space",
    "rec.sport.hockey",
    "comp.graphics",
    "talk.politics.mideast",
]

USE_REAL_DATA = True

try:
    newsgroups_train = fetch_20newsgroups(
        subset="train", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42
    )
    newsgroups_test = fetch_20newsgroups(
        subset="test", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42
    )
    texts_train_raw = newsgroups_train.data
    y_train_all = newsgroups_train.target
    texts_test = newsgroups_test.data
    y_test = newsgroups_test.target
    CLASS_NAMES = [CATEGORIES[i].split(".")[-1] for i in range(len(CATEGORIES))]
    NUM_CLASSES = len(CATEGORIES)
    print(f"20 Newsgroups loaded: {len(texts_train_raw)} train, {len(texts_test)} test")
    print(f"Classes: {CLASS_NAMES}")
except Exception as e:
    print(f"Failed to load 20 Newsgroups: {e}")
    print("Falling back to synthetic text data.")
    USE_REAL_DATA = False

In [ ]:
# Fallback: synthetic text data
if not USE_REAL_DATA:
    set_global_seed(42)
    CLASS_NAMES = ["space", "hockey", "graphics", "mideast"]
    NUM_CLASSES = 4

    # Keyword-based synthetic documents
    keyword_pools = {
        0: ["nasa", "orbit", "shuttle", "rocket", "satellite", "launch", "astronaut",
            "space", "moon", "mars", "telescope", "galaxy", "astronomy", "comet"],
        1: ["hockey", "goal", "puck", "rink", "nhl", "team", "player", "score",
            "goalie", "penalty", "period", "assist", "defenseman", "playoff"],
        2: ["graphics", "image", "render", "pixel", "opengl", "texture", "shader",
            "display", "resolution", "algorithm", "polygon", "ray", "3d", "color"],
        3: ["israel", "arab", "peace", "conflict", "government", "political", "war",
            "region", "negotiations", "settlement", "territory", "policy", "united"],
    }
    filler = ["the", "a", "is", "was", "and", "of", "to", "in", "for", "with",
              "on", "that", "this", "it", "from", "has", "have", "been", "are", "were"]

    def generate_doc(label, rng, min_len=30, max_len=80):
        length = rng.randint(min_len, max_len)
        keywords = keyword_pools[label]
        n_keywords = max(5, length // 3)
        words = list(rng.choice(keywords, n_keywords)) + list(rng.choice(filler, length - n_keywords))
        rng.shuffle(words)
        return " ".join(words)

    rng = np.random.RandomState(42)
    n_train, n_test = 1600, 400
    texts_train_raw, y_train_all = [], []
    texts_test, y_test = [], []

    for label in range(NUM_CLASSES):
        for _ in range(n_train // NUM_CLASSES):
            texts_train_raw.append(generate_doc(label, rng))
            y_train_all.append(label)
        for _ in range(n_test // NUM_CLASSES):
            texts_test.append(generate_doc(label, rng))
            y_test.append(label)

    y_train_all = np.array(y_train_all)
    y_test = np.array(y_test)
    print(f"Synthetic data created: {len(texts_train_raw)} train, {len(texts_test)} test")
    print(f"Classes: {CLASS_NAMES}")

In [ ]:
# Train/val split
texts_train, texts_val, y_train, y_val = train_test_split(
    texts_train_raw, y_train_all, test_size=0.15, random_state=42, stratify=y_train_all
)

print(f"Train: {len(texts_train)} | Val: {len(texts_val)} | Test: {len(texts_test)}")

# Class distribution
for split_name, labels in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    dist = ", ".join(f"{CLASS_NAMES[i]}: {c}" for i, c in enumerate(counts))
    print(f"  {split_name}: {dist}")

In [ ]:
# Display a few examples
print("Sample documents:\n")
for i in range(4):
    idx = i * (len(texts_train) // 4)
    text_preview = texts_train[idx][:200].replace("\n", " ")
    print(f"[{CLASS_NAMES[y_train[idx]]}] {text_preview}...")
    print()

---

## 3. Text Preprocessing Pipeline

We use `src/utils/text_preprocessing.py` for reusable text cleaning:

1. **Normalize**: lowercase, remove URLs/emails, expand contractions
2. **Tokenize**: whitespace + punctuation-aware splitting
3. **Build vocabulary**: frequency-based, with `<PAD>` and `<UNK>` tokens
4. **Convert to sequences**: integer-encoded, padded to fixed length

In [ ]:
# Step 1: Normalize all texts
texts_train_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_train]
texts_val_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_val]
texts_test_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_test]

print("Before preprocessing:")
print(f"  {texts_train[0][:150]}...")
print("\nAfter preprocessing:")
print(f"  {texts_train_clean[0][:150]}...")

In [ ]:
# Step 2: Tokenize and analyze token statistics
all_tokens = [basic_tokenize(t) for t in texts_train_clean]
lengths = [len(t) for t in all_tokens]

print(f"Token length statistics:")
print(f"  Mean:   {np.mean(lengths):.1f}")
print(f"  Median: {np.median(lengths):.1f}")
print(f"  95th:   {np.percentile(lengths, 95):.0f}")
print(f"  Max:    {np.max(lengths)}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=50, edgecolor="black", alpha=0.7)
ax.axvline(np.percentile(lengths, 95), color="red", linestyle="--", label="95th percentile")
ax.set_xlabel("Document Length (tokens)")
ax.set_ylabel("Count")
ax.set_title("Document Length Distribution")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Build vocabulary (for LSTM model later)
MAX_VOCAB_SIZE = 10000
MIN_FREQ = 2
MAX_SEQ_LEN = min(256, int(np.percentile(lengths, 95)))

vocab = build_vocab(
    texts_train_clean,
    max_vocab_size=MAX_VOCAB_SIZE,
    min_freq=MIN_FREQ,
    special_tokens=["<PAD>", "<UNK>"]
)

VOCAB_SIZE = len(vocab)
PAD_IDX = vocab["<PAD>"]

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Max sequence length: {MAX_SEQ_LEN}")
print(f"Sample vocab entries: {list(vocab.items())[:10]}")

In [ ]:
# Step 4: Convert to padded integer sequences
X_train_seq = texts_to_sequences(texts_train_clean, vocab, max_len=MAX_SEQ_LEN)
X_val_seq = texts_to_sequences(texts_val_clean, vocab, max_len=MAX_SEQ_LEN)
X_test_seq = texts_to_sequences(texts_test_clean, vocab, max_len=MAX_SEQ_LEN)

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_seq, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(f"Sequence tensor shapes:")
print(f"  Train: {X_train_tensor.shape}")
print(f"  Val:   {X_val_tensor.shape}")
print(f"  Test:  {X_test_tensor.shape}")

---

## 4. Baseline 1: TF-IDF + Logistic Regression

Our first baseline uses the classic NLP pipeline:
- **TF-IDF** vectorization to convert text to fixed-size sparse feature vectors
- **Logistic Regression** as a strong, interpretable linear classifier

This is a highly competitive baseline for text classification.

In [ ]:
# Fit TF-IDF on training data only
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(texts_train_clean)
X_val_tfidf = tfidf.transform(texts_val_clean)
X_test_tfidf = tfidf.transform(texts_test_clean)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Feature names sample: {tfidf.get_feature_names_out()[:10]}")

In [ ]:
# Train Logistic Regression
start_time = time.time()
lr_model = LogisticRegression(
    max_iter=1000, C=1.0, solver="lbfgs", multi_class="multinomial", random_state=42
)
lr_model.fit(X_train_tfidf, y_train)
lr_train_time = time.time() - start_time

# Predictions
y_pred_lr_val = lr_model.predict(X_val_tfidf)
y_pred_lr_test = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr_test)
f1_lr = f1_score(y_test, y_pred_lr_test, average="weighted")

print(f"Logistic Regression trained in {lr_train_time:.2f}s")
print(f"  Val Accuracy:  {accuracy_score(y_val, y_pred_lr_val):.4f}")
print(f"  Test Accuracy: {acc_lr:.4f}")
print(f"  Test F1 (weighted): {f1_lr:.4f}")

In [ ]:
# Classification report for Logistic Regression
print("Logistic Regression -- Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_lr_test, target_names=CLASS_NAMES))

---

## 5. Baseline 2: TF-IDF + SVM

Support Vector Machines with linear kernels are historically one of the best methods for text classification. We use `LinearSVC` for efficiency.

In [ ]:
# Train Linear SVM
start_time = time.time()
svm_model = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_train_time = time.time() - start_time

# Predictions
y_pred_svm_val = svm_model.predict(X_val_tfidf)
y_pred_svm_test = svm_model.predict(X_test_tfidf)

acc_svm = accuracy_score(y_test, y_pred_svm_test)
f1_svm = f1_score(y_test, y_pred_svm_test, average="weighted")

print(f"Linear SVM trained in {svm_train_time:.2f}s")
print(f"  Val Accuracy:  {accuracy_score(y_val, y_pred_svm_val):.4f}")
print(f"  Test Accuracy: {acc_svm:.4f}")
print(f"  Test F1 (weighted): {f1_svm:.4f}")

In [ ]:
# Classification report for SVM
print("Linear SVM -- Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_svm_test, target_names=CLASS_NAMES))

---

## 6. Deep Model: Embedding + LSTM

Now we build a deep learning text classifier:

```
Embedding(vocab_size, embed_dim)
  -> LSTM(embed_dim, hidden_dim, bidirectional=True)
  -> take final hidden state
  -> Linear(hidden_dim*2, num_classes)
```

Key design choices:
- **Bidirectional LSTM** captures context from both directions
- **Padding index** in embedding ensures `<PAD>` tokens get zero vectors
- **Dropout** for regularization

In [ ]:
class LSTMClassifier(nn.Module):
    """Bidirectional LSTM text classifier with embedding layer."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        output, (hidden, cell) = self.lstm(embedded)
        # Concatenate final hidden states from both directions
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, hidden*2)
        hidden_cat = self.dropout(hidden_cat)
        logits = self.fc(hidden_cat)  # (batch, num_classes)
        return logits


# Model hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 30
PATIENCE = 5

set_global_seed(42)
lstm_model = LSTMClassifier(
    VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES,
    num_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX
).to(device)

print(lstm_model)
n_params = sum(p.numel() for p in lstm_model.parameters())
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
# Build DataLoaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created: {len(train_loader)} train batches, "
      f"{len(val_loader)} val batches, {len(test_loader)} test batches")

In [ ]:
# Training loop for LSTM
def train_lstm(model, train_loader, val_loader, epochs, lr, patience, device):
    """Train the LSTM classifier with early stopping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )
    loss_fn = nn.CrossEntropyLoss()
    early_stop = EarlyStopping(patience=patience)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            correct += (output.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # Validate
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = loss_fn(output, y_batch)
                running_loss += loss.item() * X_batch.size(0)
                correct += (output.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        scheduler.step(val_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        if early_stop(val_loss, model):
            print(f"Early stopping at epoch {epoch}")
            early_stop.load_best(model)
            break

    elapsed = time.time() - start_time
    history["training_time"] = elapsed
    print(f"\nTraining completed in {elapsed:.1f}s")
    return history


set_global_seed(42)
lstm_model = LSTMClassifier(
    VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES,
    num_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX
).to(device)

print("=" * 60)
print("Training Embedding + LSTM Classifier")
print("=" * 60)

history_lstm = train_lstm(
    lstm_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, patience=PATIENCE, device=device
)

In [ ]:
# Plot LSTM training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history_lstm["train_loss"]) + 1)
ax1.plot(epochs_range, history_lstm["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs_range, history_lstm["val_loss"], label="Val Loss", marker="s", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("LSTM: Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history_lstm["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs_range, history_lstm["val_acc"], label="Val Acc", marker="s", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("LSTM: Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# LSTM test predictions
@torch.no_grad()
def get_predictions(model, loader, device):
    """Collect all predictions and labels from a DataLoader."""
    model.eval()
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        output = model(X_batch)
        preds = output.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())
    return np.array(all_labels), np.array(all_preds)


y_true_lstm, y_pred_lstm = get_predictions(lstm_model, test_loader, device)
acc_lstm = accuracy_score(y_true_lstm, y_pred_lstm)
f1_lstm = f1_score(y_true_lstm, y_pred_lstm, average="weighted")

print(f"LSTM Test Accuracy: {acc_lstm:.4f}")
print(f"LSTM Test F1 (weighted): {f1_lstm:.4f}")

In [ ]:
# Classification report for LSTM
print("LSTM -- Classification Report")
print("=" * 55)
print(classification_report(y_true_lstm, y_pred_lstm, target_names=CLASS_NAMES))

---

## 7. Evaluation

Side-by-side confusion matrices for all three models.

In [ ]:
# Confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

model_results = [
    (y_test, y_pred_lr_test, f"Logistic Regression\nAcc: {acc_lr:.3f} | F1: {f1_lr:.3f}"),
    (y_test, y_pred_svm_test, f"Linear SVM\nAcc: {acc_svm:.3f} | F1: {f1_svm:.3f}"),
    (y_true_lstm, y_pred_lstm, f"LSTM\nAcc: {acc_lstm:.3f} | F1: {f1_lstm:.3f}"),
]

for ax, (y_true_i, y_pred_i, title) in zip(axes, model_results):
    cm = confusion_matrix(y_true_i, y_pred_i)
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    thresh = cm.max() / 2.0
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontsize=9)

plt.suptitle("Confusion Matrices: All Approaches", fontsize=14)
plt.tight_layout()
plt.show()

---

## 8. Comparison Table

In [ ]:
# Comprehensive comparison table
lstm_time = history_lstm.get("training_time", 0)
lstm_params = sum(p.numel() for p in lstm_model.parameters())

print("\n" + "=" * 90)
print(f"{'Approach':<25} {'Test Acc':>10} {'Test F1':>10} {'Train Time':>12} {'Parameters':>14}")
print("=" * 90)

rows = [
    ("TF-IDF + LogReg", acc_lr, f1_lr, lr_train_time, "N/A (sklearn)"),
    ("TF-IDF + SVM", acc_svm, f1_svm, svm_train_time, "N/A (sklearn)"),
    ("Embedding + LSTM", acc_lstm, f1_lstm, lstm_time, f"{lstm_params:,}"),
]

for name, acc, f1, t, params in rows:
    print(f"{name:<25} {acc:>10.4f} {f1:>10.4f} {t:>10.1f}s {params:>14}")

print("=" * 90)

# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
model_names = ["LogReg", "SVM", "LSTM"]
accs = [acc_lr, acc_svm, acc_lstm]
f1s = [f1_lr, f1_svm, f1_lstm]
colors = ["tab:blue", "tab:orange", "tab:green"]

ax1.bar(model_names, accs, color=colors, edgecolor="black", alpha=0.8)
ax1.set_ylabel("Accuracy")
ax1.set_title("Test Accuracy Comparison")
ax1.set_ylim(0, 1)
for i, v in enumerate(accs):
    ax1.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)
ax1.grid(axis="y", alpha=0.3)

ax2.bar(model_names, f1s, color=colors, edgecolor="black", alpha=0.8)
ax2.set_ylabel("F1 Score (weighted)")
ax2.set_title("Test F1 Score Comparison")
ax2.set_ylim(0, 1)
for i, v in enumerate(f1s):
    ax2.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---

## 9. Error Analysis

We analyze errors from all three models to understand their failure modes:
- Which classes are hardest to classify?
- Do different models make different errors?
- Are there systematic confusion patterns?

In [ ]:
# Per-class accuracy comparison
print("Per-class Accuracy:")
print(f"{'Class':<15} {'LogReg':>10} {'SVM':>10} {'LSTM':>10}")
print("-" * 50)

for c in range(NUM_CLASSES):
    mask = y_test == c
    mask_lstm = y_true_lstm == c
    acc_lr_c = (y_pred_lr_test[mask] == c).mean()
    acc_svm_c = (y_pred_svm_test[mask] == c).mean()
    acc_lstm_c = (y_pred_lstm[mask_lstm] == c).mean()
    print(f"{CLASS_NAMES[c]:<15} {acc_lr_c:>10.3f} {acc_svm_c:>10.3f} {acc_lstm_c:>10.3f}")

In [ ]:
# Show misclassified examples from each model
def show_text_errors(texts, y_true_arr, y_pred_arr, class_names, model_name, n=5):
    """Display misclassified text examples."""
    errors = np.where(y_true_arr != y_pred_arr)[0]
    if len(errors) == 0:
        print(f"{model_name}: No errors!")
        return

    np.random.seed(42)
    selected = np.random.choice(errors, min(n, len(errors)), replace=False)

    print(f"\n{'='*60}")
    print(f"{model_name} -- Misclassified Examples ({len(errors)} total errors)")
    print(f"{'='*60}")

    for idx in selected:
        true_label = class_names[y_true_arr[idx]]
        pred_label = class_names[y_pred_arr[idx]]
        text_preview = texts[idx][:200].replace("\n", " ")
        print(f"\n  True: {true_label} | Predicted: {pred_label}")
        print(f"  Text: {text_preview}...")


# Convert texts_test to list if needed
texts_test_list = list(texts_test)

show_text_errors(texts_test_list, y_test, y_pred_lr_test, CLASS_NAMES, "Logistic Regression")
show_text_errors(texts_test_list, y_test, y_pred_svm_test, CLASS_NAMES, "Linear SVM")
show_text_errors(texts_test_list, y_true_lstm, y_pred_lstm, CLASS_NAMES, "LSTM")

In [ ]:
# Agreement analysis: where do models agree vs. disagree?
all_agree = (y_pred_lr_test == y_pred_svm_test) & (y_pred_svm_test == y_pred_lstm)
all_correct = all_agree & (y_pred_lr_test == y_test)
all_wrong = all_agree & (y_pred_lr_test != y_test)

print(f"All 3 models agree:   {all_agree.sum()} / {len(y_test)} ({all_agree.mean():.1%})")
print(f"All agree & correct:  {all_correct.sum()} / {len(y_test)} ({all_correct.mean():.1%})")
print(f"All agree & wrong:    {all_wrong.sum()} / {len(y_test)} ({all_wrong.mean():.1%})")
print(f"Models disagree:      {(~all_agree).sum()} / {len(y_test)} ({(~all_agree).mean():.1%})")

---

## 10. Conclusions

### When to Use Traditional ML vs. Deep Learning for Text Classification

| Factor | Traditional (TF-IDF + LR/SVM) | Deep Learning (LSTM) |
|--------|-------------------------------|---------------------|
| **Dataset size** | Works well with small datasets (hundreds to low thousands) | Needs more data to learn embeddings effectively |
| **Training speed** | Very fast (seconds) | Slower (minutes to hours, GPU recommended) |
| **Interpretability** | High -- can inspect feature weights | Lower -- hidden representations are opaque |
| **Feature engineering** | Manual (n-grams, TF-IDF parameters) | Learned automatically from data |
| **Sequential/contextual info** | Lost in bag-of-words representation | Captured by LSTM hidden states |
| **Transfer learning** | Limited | Can use pretrained embeddings (Word2Vec, GloVe) or transformers |
| **Deployment** | Lightweight, easy to deploy | Larger model, needs PyTorch runtime |

### Key Takeaways

- **Start with baselines.** TF-IDF + Logistic Regression is a surprisingly strong baseline for text classification. Always establish this before trying deep learning.
- **TF-IDF + SVM** is often the strongest traditional approach for text and can match or beat simple deep learning models on small to medium datasets.
- **LSTM models** shine when: (a) you have large datasets, (b) word order matters, (c) you want to leverage pretrained embeddings, or (d) you plan to extend to more complex architectures.
- **Preprocessing matters.** Both traditional and deep approaches benefit from consistent text cleaning.
- **Error analysis** reveals that all models struggle with the same ambiguous documents -- short texts, texts mixing vocabulary from multiple topics, and off-topic content.

### Next Steps

- Try pretrained word embeddings (GloVe, Word2Vec) instead of training from scratch
- Experiment with CNN-based text classifiers (TextCNN)
- Use a pretrained transformer (BERT) for fine-tuning -- see Project 03
- Try more challenging category combinations with greater overlap